In [27]:
# --- CONFIGURATION ---

# 1. Path to your trained YOLO model
MODEL_PATH = 'data/model/detect_yolo_small_v3.pt'

# 2. Path to the folder containing all your image frames
#    (Supports .jpg, .jpeg, .png image formats)
SOURCE_FRAMES_PATH = 'D:\\Recordings\\New_Recordings\\Brown_Orange_Overlay_Frames'

# 3. Name of the folder where the filtered images will be saved
DESTINATION_PATH = 'D:\\Recordings\\New_Recordings\\Brown_Orange_Overlay_Frames\\Open'

# --- END OF CONFIGURATION ---


In [7]:
if target_class_id != -1:
    # Get a list of all image files in the source directory
    image_files = glob.glob(os.path.join(SOURCE_FRAMES_PATH, '*.jpg')) + \
                  glob.glob(os.path.join(SOURCE_FRAMES_PATH, '*.jpeg')) + \
                  glob.glob(os.path.join(SOURCE_FRAMES_PATH, '*.png'))

    print(f"\nFound {len(image_files)} images to process. Starting detection...")

    images_copied = 0

    # Loop through each image with a progress bar
    for image_path in tqdm(image_files):
        # Run inference on the image
        results = model(image_path, verbose=False)  # verbose=False keeps the output clean

        # Check the results
        for result in results:
            # Get the class IDs of all detected objects in the current image
            detected_class_ids = result.boxes.cls.cpu().numpy().astype(int)

            # If our target class ID is in the list of detected IDs
            if target_class_id in detected_class_ids:
                # Copy the file to the destination folder
                file_name = os.path.basename(image_path)
                destination_file_path = os.path.join(DESTINATION_PATH, file_name)
                shutil.copy2(image_path, destination_file_path)
                images_copied += 1

                # We found the class, no need to check other detections in this image
                break

    print(f"\nProcessing complete.")
    print(f"Copied {images_copied} images containing '{TARGET_CLASS_NAME}' to the '{DESTINATION_PATH}' folder.")
else:
    print("\nSkipping image processing due to class name error.")# Load the trained YOLO model
model = YOLO(MODEL_PATH)

# Get the mapping from class names to class IDs
class_names = model.names
print(f"Model Classes: {class_names}")

# Find the specific class ID for our target class
try:
    # Create a reverse mapping from name to ID
    names_to_ids = {v: k for k, v in class_names.items()}
    target_class_id = names_to_ids[TARGET_CLASS_NAME]
    print(f"\nSuccessfully found target class: '{TARGET_CLASS_NAME}' with ID: {target_class_id}")
except KeyError:
    print(f"\nError: The class name '{TARGET_CLASS_NAME}' was not found in the model.")
    print("Please check the 'TARGET_CLASS_NAME' in Cell 2 and ensure it matches one of the model's classes.")
    target_class_id = -1 # Set to an invalid ID to stop execution

Model Classes: {0: 'bread-bag-closed', 1: 'bread-bag-opened'}

Successfully found target class: 'bread-bag-opened' with ID: 1
